# Resume Week 2, then run all of Week 3

This notebook is the restart-safe continuation entry point for the complete team benchmark. It first inventories the full-run artifacts already saved in Drive, reruns only incomplete Week 2 training jobs, and then performs the full Week 3 analysis.

Week 3 includes all four 5×5 matrices (100 cells), Δ/w/Score, the lambda-weight ranking sweep, per-shift tables, descriptive shift attribution, bootstrap confidence intervals, paired shift-vector tests, and the two-architecture normal-vs-abnormal ablation.

A single Colab GPU cannot safely train several models simultaneously. The notebook runs GPU jobs sequentially but controls the entire queue automatically. Completed full runs are reused; a checkpoint without final test metrics is treated as incomplete and rerun.

**Run every cell from the top.** Use a GPU runtime for the full experiment. Do not mark the benchmark complete unless the final audit says `ALL_REQUIRED_ARTIFACTS_COMPLETE`.


In [ ]:
#@title 1. Controls
MODE = "full"  #@param ["smoke", "full"]
RUN_MISSING_WEEK2 = True  #@param {type:"boolean"}
RUN_WEEK3_FIVE_CLASS = True  #@param {type:"boolean"}
RUN_WEEK3_TWO_CLASS = True  #@param {type:"boolean"}
REQUIRE_ALL_FIVE_DATASETS = True  #@param {type:"boolean"}
STAGE_NPY_SIGNALS_TO_LOCAL_DISK = True  #@param {type:"boolean"}
BOOTSTRAP_REPLICATES = 1000  #@param {type:"integer"}
PROJECT_ROOT_OVERRIDE = ""  #@param {type:"string"}
SHIFT_TABLE_OVERRIDE = ""  #@param {type:"string"}
SEED = 42
DATASETS = ["ptbxl", "cpsc2018", "georgia", "mimic_iv", "code_ii"]
ARCHITECTURES = ["inception_time", "resnet1d", "transformer", "ecg_fm"]
BINARY_ARCHITECTURES = ["ecg_fm", "inception_time"]
if MODE not in {"smoke", "full"}:
    raise ValueError("MODE must be smoke or full")
if BOOTSTRAP_REPLICATES < 0:
    raise ValueError("BOOTSTRAP_REPLICATES cannot be negative")
print({"mode": MODE, "datasets": DATASETS, "architectures": ARCHITECTURES})


In [ ]:
#@title 2. Mount Drive, install the current repository, and verify its version
from google.colab import drive, userdata
drive.mount("/content/drive")

import base64
import datetime as dt
import json
import os
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path

REPOSITORY_URL = "https://github.com/tanushappapogu-max/ecg-generalization-benchmark.git"
WORKSPACE = Path("/content/ecg-generalization-benchmark")
try:
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = None
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
if github_token:
    encoded = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
    git_env.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {encoded}",
    })
if WORKSPACE.exists() and not (WORKSPACE / ".git").is_dir():
    raise RuntimeError(f"Partial checkout at {WORKSPACE}; restart the Colab session before retrying")
if not WORKSPACE.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY_URL, str(WORKSPACE)], env=git_env)
else:
    subprocess.check_call(["git", "-C", str(WORKSPACE), "pull", "--ff-only"], env=git_env)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", f"{WORKSPACE}[train,test]"] )
subprocess.check_call(["git", "-C", str(WORKSPACE), "fetch", "origin", "main"], env=git_env)
local_commit = subprocess.check_output(["git", "-C", str(WORKSPACE), "rev-parse", "HEAD"], text=True).strip()
remote_commit = subprocess.check_output(["git", "-C", str(WORKSPACE), "rev-parse", "origin/main"], text=True).strip()
if local_commit != remote_commit:
    raise RuntimeError(f"Old repository version: local={local_commit[:12]} main={remote_commit[:12]}")
required = [
    "src/training/baseline_pipeline.py", "src/training/ecg_fm_pipeline.py",
    "src/training/binary_ablation_pipeline.py", "src/evaluation/baseline_matrix.py",
    "src/evaluation/ecg_fm_matrix.py", "src/evaluation/binary_matrix.py",
    "src/evaluation/week3_analysis.py",
]
missing = [name for name in required if not (WORKSPACE / name).is_file()]
if missing:
    raise RuntimeError("GitHub main is missing required continuation code: " + ", ".join(missing))
sys.path.insert(0, str(WORKSPACE))
print({"repository_status": "CURRENT_WITH_GITHUB_MAIN", "commit": local_commit})


In [ ]:
#@title 3. Locate the reorganized Drive project and prepare dataset roots
PROJECT_FOLDER_NAME = "LSTS ECG Generalization Benchmark — START HERE"
def locate_project_root():
    if PROJECT_ROOT_OVERRIDE:
        path = Path(PROJECT_ROOT_OVERRIDE)
        if not path.is_dir():
            raise FileNotFoundError(path)
        return path
    direct = [Path("/content/drive/MyDrive") / PROJECT_FOLDER_NAME, Path("/content/drive/Shareddrives") / PROJECT_FOLDER_NAME]
    for path in direct:
        if path.is_dir():
            return path
    for root in [Path("/content/drive/Shareddrives"), Path("/content/drive/MyDrive")]:
        if not root.exists():
            continue
        for base, directories, _ in os.walk(root):
            if PROJECT_FOLDER_NAME in directories:
                return Path(base) / PROJECT_FOLDER_NAME
    raise FileNotFoundError(f"Cannot locate {PROJECT_FOLDER_NAME!r}; set PROJECT_ROOT_OVERRIDE")

PROJECT_ROOT = locate_project_root()
DATASETS_ROOT = PROJECT_ROOT / "01_DATASETS"
CODE_ROOT = PROJECT_ROOT / "02_CODE_AND_NOTEBOOKS"
RESULTS_ROOT = PROJECT_ROOT / "03_RESULTS_AND_MODEL_OUTPUTS"
for path in [DATASETS_ROOT, CODE_ROOT, RESULTS_ROOT]:
    if not path.is_dir():
        raise FileNotFoundError(f"Missing organized project directory: {path}")

prefixes = {
    "ptbxl": "01_PTB_XL", "cpsc2018": "02_CPSC_2018",
    "georgia": "03_GEORGIA_12_LEAD", "mimic_iv": "04_MIMIC_IV_ECG",
    "code_ii": "05_CODE_15_PERCENT",
}
def dataset_folder(name):
    matches = sorted(path for path in DATASETS_ROOT.glob(prefixes[name] + "*") if path.is_dir())
    return matches[0] if matches else DATASETS_ROOT / prefixes[name]
DATASET_FOLDERS = {name: dataset_folder(name) for name in DATASETS}

MANIFEST_ROOT = Path("/content/ecg-week2-manifests")
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)
drive_manifest_root = CODE_ROOT / "03_ECG_FM_TRAINING_AND_EVALUATION" / "manifests"
manifest_sources = {}
for name in DATASETS:
    filename = f"{name}_week2.csv"
    candidates = [drive_manifest_root / filename, DATASET_FOLDERS[name] / filename, WORKSPACE / "data" / "week2" / filename]
    source = next((path for path in candidates if path.is_file()), None)
    if source:
        shutil.copy2(source, MANIFEST_ROOT / filename)
        manifest_sources[name] = source

LOCAL_DATA_ROOT = Path("/content/ecg-benchmark-data")
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
def stage_npy(name):
    source = DATASET_FOLDERS[name]
    source_signals = source / "signals"
    if not source_signals.is_dir():
        return source
    if not STAGE_NPY_SIGNALS_TO_LOCAL_DISK:
        return source
    destination = LOCAL_DATA_ROOT / name
    if not (destination / "signals").is_dir():
        destination.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source_signals, destination / "signals", dirs_exist_ok=True)
    return destination

def prepare_georgia():
    source = DATASET_FOLDERS["georgia"]
    if (source / "signals").is_dir():
        return stage_npy("georgia")
    destination = LOCAL_DATA_ROOT / "georgia"
    if (destination / "signals").is_dir():
        return destination
    parts = sorted(source.glob("georgia_signals.tar.gz.part-*"))
    if not parts:
        return source
    archive = Path("/content/georgia_signals.tar.gz")
    with archive.open("wb") as output:
        for part in parts:
            with part.open("rb") as handle:
                shutil.copyfileobj(handle, output, length=8 * 1024 * 1024)
    destination.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, "r:gz") as handle:
        handle.extractall(destination, filter="data")
    return destination

def prepare_mimic():
    source = DATASET_FOLDERS["mimic_iv"]
    direct_candidates = [source / "mimic_50k_waveforms", source]
    for candidate in direct_candidates:
        if (candidate / "files").is_dir():
            return candidate
    destination = LOCAL_DATA_ROOT / "mimic_iv"
    for candidate in [destination / "mimic_50k_waveforms", destination]:
        if (candidate / "files").is_dir():
            return candidate
    shard_root = source / "mimic_50k_waveforms"
    shards = sorted(shard_root.glob("mimic_50k_waveforms_*-of-050.tar.gz"))
    if len(shards) != 50:
        return source
    destination.mkdir(parents=True, exist_ok=True)
    for archive in shards:
        with tarfile.open(archive, "r:gz") as handle:
            handle.extractall(destination, filter="data")
    overlay = shard_root / "mimic_50k_v3_replacement_overlay"
    if overlay.is_dir():
        for archive in sorted(overlay.glob("*.tar.gz")):
            with tarfile.open(archive, "r:gz") as handle:
                handle.extractall(destination, filter="data")
    for candidate in [destination / "mimic_50k_waveforms", destination]:
        if (candidate / "files").is_dir():
            return candidate
    return destination

SIGNAL_ROOTS = {
    "ptbxl": stage_npy("ptbxl"), "cpsc2018": stage_npy("cpsc2018"),
    "georgia": prepare_georgia(), "mimic_iv": prepare_mimic(),
    "code_ii": stage_npy("code_ii"),
}
print({"project_root": str(PROJECT_ROOT), "manifest_sources": {k: str(v) for k, v in manifest_sources.items()}, "signal_roots": {k: str(v) for k, v in SIGNAL_ROOTS.items()}})


In [ ]:
#@title 4. Fast readiness check and exact resume inventory
import numpy as np
import pandas as pd
from src.data.week2_manifest import validate_canonical_manifest

def waveform_exists(root, row):
    path = root / str(row["signal_path"])
    storage = str(row["storage"])
    if storage == "npy":
        return path.is_file()
    if storage == "wfdb":
        base = path.with_suffix("") if path.suffix in {".hea", ".dat"} else path
        return base.with_suffix(".hea").is_file() and base.with_suffix(".dat").is_file()
    return False

readiness_rows = []
for name in DATASETS:
    manifest_path = MANIFEST_ROOT / f"{name}_week2.csv"
    root = SIGNAL_ROOTS[name]
    try:
        if not manifest_path.is_file():
            raise FileNotFoundError(f"missing frozen manifest {manifest_path.name}")
        manifest = validate_canonical_manifest(pd.read_csv(manifest_path, low_memory=False))
        if not root.exists():
            raise FileNotFoundError(f"missing signal root {root}")
        sample_count = min(100, len(manifest))
        positions = np.linspace(0, len(manifest) - 1, sample_count, dtype=int)
        sampled = manifest.iloc[np.unique(positions)]
        missing_sample = sum(not waveform_exists(root, row) for _, row in sampled.iterrows())
        if missing_sample:
            raise FileNotFoundError(f"{missing_sample}/{len(sampled)} sampled waveforms are missing")
        readiness_rows.append({"dataset": name, "preflight_pass": True, "records": len(manifest), "reason": "manifest valid; sampled waveform paths exist"})
    except Exception as exc:
        readiness_rows.append({"dataset": name, "preflight_pass": False, "records": 0, "reason": str(exc)})
readiness = pd.DataFrame(readiness_rows)
display(readiness)
READY_DATASETS = readiness.loc[readiness["preflight_pass"], "dataset"].tolist()
BLOCKED_DATASETS = readiness.loc[~readiness["preflight_pass"], "dataset"].tolist()

MASTER_RUN_ROOT = RESULTS_ROOT / "MASTER_COLAB_RUNS" / MODE
FIVE_CLASS_RUNS = MASTER_RUN_ROOT / "01_FIVE_CLASS_CHECKPOINTS"
FIVE_CLASS_MATRICES = MASTER_RUN_ROOT / "02_FIVE_CLASS_MATRICES"
BINARY_RUNS = MASTER_RUN_ROOT / "03_TWO_CLASS_CHECKPOINTS"
BINARY_MATRICES = MASTER_RUN_ROOT / "04_TWO_CLASS_MATRICES"
WEEK3_ANALYSIS = MASTER_RUN_ROOT / "05_WEEK3_POOLED_ANALYSIS"
STATUS_PATH = MASTER_RUN_ROOT / "resume_week2_week3_status.json"
MASTER_RUN_ROOT.mkdir(parents=True, exist_ok=True)
EXPECTED_TRAINING_STATUS = "SMOKE_PASS" if MODE == "smoke" else "COMPLETE"

def read_json(path):
    try:
        return json.loads(path.read_text())
    except Exception:
        return {}
def training_complete(output_dir):
    metrics = read_json(output_dir / "test_metrics.json")
    return (output_dir / "best_checkpoint.pt").is_file() and (output_dir / "test_predictions.csv").is_file() and metrics.get("status") == EXPECTED_TRAINING_STATUS
def matrix_complete(output_dir, architecture):
    summary = read_json(output_dir / "matrix_summary.json")
    filename = "ecg_fm_five_label_matrix_long.csv" if architecture == "ecg_fm" else f"{architecture}_five_label_matrix_long.csv"
    matrix_path = output_dir / filename
    checkpoints = [FIVE_CLASS_RUNS / architecture / dataset / "best_checkpoint.pt" for dataset in DATASETS]
    fresh = matrix_path.is_file() and all(path.is_file() and matrix_path.stat().st_mtime >= path.stat().st_mtime for path in checkpoints)
    return fresh and summary.get("expected_cells") == 25 and summary.get("completed_cells") == 25 and summary.get("blocked_cells") == 0
inventory = []
for architecture in ARCHITECTURES:
    for dataset in DATASETS:
        output_dir = FIVE_CLASS_RUNS / architecture / dataset
        inventory.append({"architecture": architecture, "dataset": dataset, "week2_status": "COMPLETE_REUSE" if training_complete(output_dir) else "MISSING_OR_INCOMPLETE", "output_dir": str(output_dir)})
display(pd.DataFrame(inventory))
print({"ready_datasets": READY_DATASETS, "blocked_datasets": BLOCKED_DATASETS, "output_root": str(MASTER_RUN_ROOT)})


In [ ]:
#@title 5. Run the restart-safe Week 2 → Week 3 queue
if REQUIRE_ALL_FIVE_DATASETS and BLOCKED_DATASETS:
    raise RuntimeError("Official run stopped before spending GPU time. Fix these datasets first: " + ", ".join(BLOCKED_DATASETS))
RUN_DATASETS = DATASETS if REQUIRE_ALL_FIVE_DATASETS else READY_DATASETS
if not RUN_DATASETS:
    raise RuntimeError("No datasets passed preflight")

subprocess.check_call([sys.executable, "-m", "pytest", "-q", "tests/test_inception_time.py", "tests/test_resnet1d.py", "tests/test_transformer1d.py", "tests/test_ecg_fm_model.py", "tests/test_composite_score.py", "tests/test_binary_matrix.py", "tests/test_week3_analysis.py"], cwd=WORKSPACE)
print("Targeted architecture and Week 3 unit tests: PASS")

import torch
if MODE == "full" and not torch.cuda.is_available():
    raise RuntimeError("Full training requires a GPU runtime: Runtime > Change runtime type > GPU")

job_status = []
def save_status():
    STATUS_PATH.write_text(json.dumps(job_status, indent=2) + "\n")
def run_command(label, command):
    print("\nRUNNING", label)
    print(" ".join(map(str, command)))
    started = dt.datetime.now(dt.timezone.utc)
    row = {"label": label, "status": "RUNNING", "started_utc": started.isoformat()}
    job_status.append(row)
    save_status()
    try:
        subprocess.check_call(list(map(str, command)), cwd=WORKSPACE)
        row["status"] = "COMPLETE"
    except Exception as exc:
        row["status"] = "FAILED"
        row["detail"] = repr(exc)
        raise
    finally:
        row["finished_utc"] = dt.datetime.now(dt.timezone.utc).isoformat()
        save_status()
def record_skip(label, reason):
    row = {"label": label, "status": "SKIPPED_COMPLETE", "detail": reason}
    job_status.append(row)
    save_status()
    print(row)
def signal_root_arguments():
    result = []
    for dataset in RUN_DATASETS:
        result += ["--signal-root", f"{dataset}={SIGNAL_ROOTS[dataset]}"]
    return result

ECG_FM_CHECKPOINT = Path("/content/checkpoints/ecg-fm/mimic_iv_ecg_physionet_pretrained.pt")
def ensure_ecg_fm():
    global ECG_FM_CHECKPOINT
    fairseq = Path("/content/fairseq-signals")
    if not fairseq.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/Jwoo5/fairseq-signals.git", str(fairseq)])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(fairseq)], env={**os.environ, "MAX_JOBS": "2"})
    from huggingface_hub import hf_hub_download
    ECG_FM_CHECKPOINT = Path(hf_hub_download(repo_id="wanglab/ecg-fm", filename="mimic_iv_ecg_physionet_pretrained.pt", local_dir="/content/checkpoints/ecg-fm"))
    if ECG_FM_CHECKPOINT.stat().st_size < 100_000_000:
        raise RuntimeError("ECG-FM checkpoint is unexpectedly small")

need_ecg_fm = any(not training_complete(FIVE_CLASS_RUNS / "ecg_fm" / dataset) for dataset in RUN_DATASETS) or RUN_WEEK3_FIVE_CLASS or RUN_WEEK3_TWO_CLASS
if need_ecg_fm:
    ensure_ecg_fm()

if RUN_MISSING_WEEK2:
    for architecture in ARCHITECTURES:
        for dataset in RUN_DATASETS:
            output_dir = FIVE_CLASS_RUNS / architecture / dataset
            label = f"week2::{architecture}::{dataset}"
            if training_complete(output_dir):
                record_skip(label, "validated checkpoint + final held-out metrics already exist")
                continue
            if architecture == "ecg_fm":
                command = [sys.executable, "-m", "src.training.ecg_fm_pipeline", "--dataset", dataset, "--manifest", MANIFEST_ROOT / f"{dataset}_week2.csv", "--signal-root", SIGNAL_ROOTS[dataset], "--pretrained-checkpoint", ECG_FM_CHECKPOINT, "--output-dir", output_dir, "--seed", str(SEED), "--freeze-feature-extractor"]
                command += ["--epochs", "1", "--patience", "1", "--max-records-per-split", "128", "--batch-size", "2", "--gradient-accumulation-steps", "1", "--smoke-test"] if MODE == "smoke" else ["--epochs", "50", "--patience", "10", "--batch-size", "4", "--gradient-accumulation-steps", "8", "--learning-rate", "1e-6"]
            else:
                command = [sys.executable, "-m", "src.training.baseline_pipeline", "--architecture", architecture, "--dataset", dataset, "--manifest", MANIFEST_ROOT / f"{dataset}_week2.csv", "--signal-root", SIGNAL_ROOTS[dataset], "--output-dir", output_dir, "--seed", str(SEED)]
                if MODE == "smoke":
                    command += ["--epochs", "1", "--patience", "1", "--max-records-per-split", "128", "--batch-size", "4", "--gradient-accumulation-steps", "1", "--smoke-test", "--no-mixed-precision"]
                    if architecture == "inception_time": command += ["--inception-channels", "4", "--inception-depth", "3"]
                    if architecture == "resnet1d": command += ["--resnet-base-channels", "4", "--resnet-blocks", "1", "1", "1", "1"]
                    if architecture == "transformer": command += ["--transformer-patch-size", "100", "--transformer-embed-dim", "16", "--transformer-heads", "2", "--transformer-layers", "1", "--transformer-feedforward-dim", "32"]
                else:
                    command += ["--epochs", "50", "--patience", "10", "--batch-size", "16" if architecture == "transformer" else "32", "--learning-rate", "3e-4" if architecture == "transformer" else "1e-3"]
            run_command(label, command)
            if not training_complete(output_dir):
                raise RuntimeError(f"{label} exited without a complete checkpoint/test-metrics bundle")

missing_week2 = [(a, d) for a in ARCHITECTURES for d in RUN_DATASETS if not training_complete(FIVE_CLASS_RUNS / a / d)]
if missing_week2:
    raise RuntimeError(f"Week 3 cannot start; incomplete Week 2 runs: {missing_week2}")

if RUN_WEEK3_FIVE_CLASS:
    for architecture in ARCHITECTURES:
        output_dir = FIVE_CLASS_MATRICES / architecture
        label = f"week3_matrix::{architecture}"
        if MODE == "full" and matrix_complete(output_dir, architecture):
            record_skip(label, "validated 25/25 matrix already exists")
            continue
        if architecture == "ecg_fm":
            command = [sys.executable, "-m", "src.evaluation.ecg_fm_matrix", "--source-runs-root", FIVE_CLASS_RUNS / architecture, "--manifest-root", MANIFEST_ROOT, "--pretrained-checkpoint", ECG_FM_CHECKPOINT, "--output-dir", output_dir, "--datasets", *RUN_DATASETS, "--seed", str(SEED), *signal_root_arguments()]
        else:
            command = [sys.executable, "-m", "src.evaluation.baseline_matrix", "--architecture", architecture, "--source-runs-root", FIVE_CLASS_RUNS / architecture, "--manifest-root", MANIFEST_ROOT, "--output-dir", output_dir, "--datasets", *RUN_DATASETS, "--seed", str(SEED), *signal_root_arguments()]
        if MODE == "smoke": command += ["--max-records-per-target", "128", "--batch-size", "2", "--no-mixed-precision"]
        else: command += ["--fail-on-missing"]
        run_command(label, command)

    matrix_arguments = []
    for architecture in ARCHITECTURES:
        filename = "ecg_fm_five_label_matrix_long.csv" if architecture == "ecg_fm" else f"{architecture}_five_label_matrix_long.csv"
        matrix_arguments += ["--matrix", f"{architecture}={FIVE_CLASS_MATRICES / architecture / filename}"]
    shift_table = Path(SHIFT_TABLE_OVERRIDE) if SHIFT_TABLE_OVERRIDE else WORKSPACE / "results" / "shift_metadata" / "shift_vectors.csv"
    analysis_summary = read_json(WEEK3_ANALYSIS / "week3_analysis_summary.json")
    analysis_path = WEEK3_ANALYSIS / "week3_analysis_summary.json"
    matrix_paths_for_freshness = [Path(value.split("=", 1)[1]) for flag, value in zip(matrix_arguments[::2], matrix_arguments[1::2])]
    analysis_fresh = analysis_path.is_file() and all(analysis_path.stat().st_mtime >= path.stat().st_mtime for path in matrix_paths_for_freshness)
    if MODE == "full" and analysis_fresh and analysis_summary.get("status") == "COMPLETE" and analysis_summary.get("bootstrap_replicates") == BOOTSTRAP_REPLICATES:
        record_skip("week3::pooled_analysis", "validated pooled analysis already exists")
    else:
        command = [sys.executable, "-m", "src.evaluation.week3_analysis", *matrix_arguments, "--shift-table", shift_table, "--output-dir", WEEK3_ANALYSIS, "--datasets", *RUN_DATASETS, "--expected-architectures", *ARCHITECTURES, "--bootstrap-replicates", str(50 if MODE == "smoke" else BOOTSTRAP_REPLICATES), "--seed", str(SEED)]
        run_command("week3::pooled_analysis", command)

if RUN_WEEK3_TWO_CLASS:
    for architecture in BINARY_ARCHITECTURES:
        for dataset in RUN_DATASETS:
            output_dir = BINARY_RUNS / architecture / dataset
            label = f"week3_binary_train::{architecture}::{dataset}"
            if training_complete(output_dir):
                record_skip(label, "validated binary checkpoint + held-out metrics already exist")
                continue
            command = [sys.executable, "-m", "src.training.binary_ablation_pipeline", "--architecture", architecture, "--manifest", MANIFEST_ROOT / f"{dataset}_week2.csv", "--signal-root", SIGNAL_ROOTS[dataset], "--output-dir", output_dir, "--seed", str(SEED)]
            if architecture == "ecg_fm": command += ["--pretrained-checkpoint", ECG_FM_CHECKPOINT, "--learning-rate", "1e-6", "--batch-size", "4", "--gradient-accumulation-steps", "8"]
            else: command += ["--learning-rate", "1e-3", "--batch-size", "32", "--gradient-accumulation-steps", "1"]
            command += ["--epochs", "1", "--patience", "1", "--max-records-per-split", "128", "--smoke-test"] if MODE == "smoke" else ["--epochs", "50", "--patience", "10"]
            run_command(label, command)
            if not training_complete(output_dir): raise RuntimeError(f"{label} did not emit a complete result bundle")
    binary_summary = read_json(BINARY_MATRICES / "binary_matrix_summary.json")
    binary_matrix_path = BINARY_MATRICES / "binary_matrix_long.csv"
    binary_checkpoints = [BINARY_RUNS / architecture / dataset / "best_checkpoint.pt" for architecture in BINARY_ARCHITECTURES for dataset in DATASETS]
    binary_fresh = binary_matrix_path.is_file() and all(path.is_file() and binary_matrix_path.stat().st_mtime >= path.stat().st_mtime for path in binary_checkpoints)
    binary_complete = binary_fresh and binary_summary.get("expected_cells") == 50 and binary_summary.get("completed_cells") == 50 and binary_summary.get("blocked_cells") == 0
    if MODE == "full" and binary_complete:
        record_skip("week3::binary_matrices", "validated 50/50 two-architecture matrix already exists")
    else:
        command = [sys.executable, "-m", "src.evaluation.binary_matrix", "--manifest-root", MANIFEST_ROOT, "--source-runs-root", BINARY_RUNS, "--pretrained-checkpoint", ECG_FM_CHECKPOINT, "--output-dir", BINARY_MATRICES, "--datasets", *RUN_DATASETS, "--architectures", *BINARY_ARCHITECTURES, "--seed", str(SEED), *signal_root_arguments()]
        for architecture in BINARY_ARCHITECTURES:
            filename = "ecg_fm_five_label_matrix_long.csv" if architecture == "ecg_fm" else f"{architecture}_five_label_matrix_long.csv"
            command += ["--five-label-matrix", f"{architecture}={FIVE_CLASS_MATRICES / architecture / filename}"]
        if MODE == "smoke": command += ["--max-records-per-target", "128", "--batch-size", "2"]
        run_command("week3::binary_matrices", command)
print("QUEUE FINISHED. Run the final audit cell.")


In [ ]:
#@title 6. Final completion audit and files to show the team
audit = []
for architecture in ARCHITECTURES:
    for dataset in DATASETS:
        audit.append({"stage": "week2_training", "architecture": architecture, "dataset": dataset, "complete": training_complete(FIVE_CLASS_RUNS / architecture / dataset), "path": str(FIVE_CLASS_RUNS / architecture / dataset)})
for architecture in ARCHITECTURES:
    audit.append({"stage": "week3_five_class_matrix", "architecture": architecture, "dataset": "ALL_5x5", "complete": matrix_complete(FIVE_CLASS_MATRICES / architecture, architecture), "path": str(FIVE_CLASS_MATRICES / architecture)})
analysis_summary = read_json(WEEK3_ANALYSIS / "week3_analysis_summary.json")
audit.append({"stage": "week3_pooled_analysis", "architecture": "ALL", "dataset": "ALL", "complete": analysis_summary.get("status") == "COMPLETE", "path": str(WEEK3_ANALYSIS)})
binary_summary = read_json(BINARY_MATRICES / "binary_matrix_summary.json")
audit.append({"stage": "week3_binary_ablation", "architecture": "ecg_fm+inception_time", "dataset": "ALL_5x5", "complete": binary_summary.get("completed_cells") == 50 and binary_summary.get("blocked_cells") == 0, "path": str(BINARY_MATRICES)})
audit_frame = pd.DataFrame(audit)
display(audit_frame)
audit_frame.to_csv(MASTER_RUN_ROOT / "FINAL_COMPLETION_AUDIT.csv", index=False)
if audit_frame["complete"].all():
    print("ALL_REQUIRED_ARTIFACTS_COMPLETE")
    print("Five-class pooled results:", WEEK3_ANALYSIS / "pooled_five_label_results.csv")
    print("Bootstrap intervals:", WEEK3_ANALYSIS / "bootstrap_95ci_all_cells.csv")
    print("Two-class gap result:", BINARY_MATRICES / "two_class_gap_summary.csv")
else:
    print("NOT COMPLETE — only rows marked True may be crossed off.")
    display(audit_frame.loc[~audit_frame["complete"]])
